# Big Mart Sales Prediction — EDA & Model Training
**Owner:** Ahasna  
Follow the steps in order. Each markdown heading matches a step in the group project guide.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

## 2. Load the Dataset
Make sure `Train.csv` is inside the `data/` folder first (see `data/README.md`).

In [ ]:
df = pd.read_csv('../data/Train.csv')
df.head()

## 3. Explore the Data (EDA)
Write your answers/observations for each of these in a markdown cell — you'll need them for the report.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Look for inconsistent category labels, e.g. Item_Fat_Content
df['Item_Fat_Content'].value_counts()

**Your notes here:** dataset source, number of records, number of features, target variable, feature descriptions, data types, missing values, duplicates, data quality issues.

## 4. Feature Engineering
At least 5–6 techniques are required. Explain *why* for each one in the report.

### 4.1 Handle Missing Values

In [ ]:
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].mean())
df['Outlet_Size'] = df['Outlet_Size'].fillna(df['Outlet_Size'].mode()[0])

### 4.2 Fix & Encode Categorical Variables

In [ ]:
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'
})

categorical_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Size',
                    'Outlet_Location_Type', 'Outlet_Type']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

### 4.3 Feature Scaling

In [ ]:
numeric_cols = ['Item_Weight', 'Item_Visibility', 'Item_MRP']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

### 4.4 Create a New Feature — Outlet_Age

In [ ]:
df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']
# 2013 is the year this dataset was collected — see data/README.md

### 4.5 Outlier Treatment

In [ ]:
Q1 = df['Item_Visibility'].quantile(0.25)
Q3 = df['Item_Visibility'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
df['Item_Visibility'] = np.where(df['Item_Visibility'] > upper_limit, upper_limit, df['Item_Visibility'])

### 4.6 Binning (optional bonus)

In [ ]:
# df['MRP_Range'] = pd.cut(df['Item_MRP'], bins=4,
#                          labels=['Low', 'Medium', 'High', 'Very High'])

## 5. Split the Data

In [ ]:
X = df.drop(['Item_Outlet_Sales', 'Item_Identifier', 'Outlet_Identifier',
             'Outlet_Establishment_Year'], axis=1, errors='ignore')
y = df['Item_Outlet_Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## 6. Train Machine Learning Models

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

## 7. Evaluate & Compare Models

In [ ]:
def evaluate(name, model):
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))  # works on every sklearn version
    r2 = r2_score(y_test, preds)
    print(f'{name}: RMSE = {rmse:.2f}, R2 = {r2:.4f}')
    return rmse, r2

evaluate('Linear Regression', lr_model)
evaluate('Random Forest', rf_model)

## 8. Save the Best Model
Swap `rf_model` for whichever model actually scored best above.

In [ ]:
best_model = rf_model  # TODO: confirm this is the winner

joblib.dump(best_model, 'model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(encoders, 'encoder.joblib')

print('Saved model.joblib, scaler.joblib, encoder.joblib')
print('Copy these 3 files to backend/model/ and send them to KaveeN.')